In [ ]:
!pip install easyocr
import pandas as pd
import easyocr
import os
import tqdm
import tqdm.notebook
from tqdm.notebook import tqdm

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', None)

from google.colab import drive
drive.mount('/content/drive')

os.chdir("/content/drive/MyDrive/Daten")


In [ ]:
# Define the path to the images folder
images_root_path = 'Bilder'

# Initialize the EasyOCR reader
reader = easyocr.Reader(['de'])

# Initialize a dictionary to store OCR results
ocr_results = {}

# Loop through each subfolder in the images folder
for root, dirs, files in os.walk(images_root_path):
    for file in tqdm(files, desc=f"Processing images in {root}"):
        if file.endswith(('.jpg', '.jpeg', '.png')):  # Add more image file extensions if needed
            image_path = os.path.join(root, file)
            author = os.path.basename(root)
            image_id, _ = os.path.splitext(file)

            # Read the image using EasyOCR
            text = reader.readtext(image_path)

            # Extracted text as a single string
            extracted_text = ' '.join([line[1] for line in text])

            # Store the result in the dictionary
            ocr_results[(author, image_id)] = extracted_text

In [ ]:
ocr_results

In [ ]:
# Aus ocr-dictionary wird dataframe
ocr = pd.DataFrame(ocr_results.items(), columns=["author_filenameimage", "ocr_text"])

In [ ]:
#In einer Variable sind sowohl author als auch filename, deshalb werden sie getrennt
ocr["author_filenameimage"] = ocr["author_filenameimage"].astype(str)
ocr[["author","filename_image"]] = ocr["author_filenameimage"].str.split(", ", expand=True)

In [ ]:
# Bereinigung
ocr["author"] = ocr["author"].str.replace("('","")
ocr["author"] = ocr["author"].str.replace("'","")
ocr["filename_image"] = ocr["filename_image"].str.replace("'","")
ocr["filename_image"] = ocr["filename_image"].str.replace(")","")

In [ ]:
ocr[["id_scraping","nutzlos_1"]] = ocr["author"].str.split("_", expand=True)
ocr["id_scraping"] = ocr["id_scraping"].astype(int)
ocr = ocr.drop(["nutzlos_1","author_filenameimage","author"], axis=1)

In [ ]:
ocr_alt = pd.read_csv("ocr_colab.csv")

In [ ]:
ocr = pd.concat([ocr,ocr_alt])

In [ ]:
ocr["id_scraping"].dtype

In [ ]:
ocr.to_csv("ocr_colab.csv", index=False)